In [1]:
!python -m pip install tree-sitter tree-sitter-python qdrant-client fastembed


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## AST 递归提取与分块

In [2]:
from dataclasses import dataclass
from typing import List, Optional
from tree_sitter import Language, Parser
import tree_sitter_python as tspython


# 构建语言映射表
LANGUAGES = {
    "python": Language(tspython.language()),
}


@dataclass
class CodeChunk:
    file_path: str
    chunk_type: str  # 'function', 'class', 'module'
    name: str  # 函数名 / 类名
    start_line: int
    end_line: int
    content: str
    context: str  # 文件路径、父类/命名空间、依赖等元数据


class CodeASTSplitter:
    def __init__(self, language: str, max_chunk_size: int = 1500):
        self.max_chunk_size = max_chunk_size
        lang_obj = LANGUAGES.get(language)
        if lang_obj:
            self.parser = Parser(lang_obj)
        else:
            self.parser = None

        # 定义需要提取的目标语法节点类型
        self.target_node_types = {
            "python": {
                "function_definition",
                "class_definition",
                "async_function_definition",
            },
            "typescript": {
                "function_declaration",
                "class_declaration",
                "method_definition",
                "arrow_function",
            },
            "java": {
                "method_declaration",
                "class_declaration",
                "interface_declaration",
            },
        }.get(language, {"function_definition", "class_definition"})

    def split_file(self, file_path: str, code_content: str) -> List[CodeChunk]:
        # 1. 如果没有对应的 AST 解析器，直接当作纯文本按行切分
        if self.parser is None:
            return self._fallback_line_split(file_path, code_content, header="")
        
        # 2. 进行 AST 解析
        tree = self.parser.parse(bytes(code_content, "utf8"))
        root_node = tree.root_node
        chunks: List[CodeChunk] = []

        # 提取文件头部的 import/包声明作为通用上下文
        header_context = self._extract_header_imports(root_node, code_content)

        def traverse(node, scope_prefix=""):
            # 命中目标函数或类
            if node.type in self.target_node_types:
                node_text = code_content[node.start_byte : node.end_byte]
                name = self._get_node_name(node, code_content)
                current_scope = f"{scope_prefix}.{name}" if scope_prefix else name

                # 如果代码块过长，继续递归内部节点；否则作为一个独立 chunk
                if len(node_text) > self.max_chunk_size and node.children:
                    for child in node.children:
                        traverse(child, current_scope)
                else:
                    context = f"// File: {file_path}\n// Scope: {current_scope}\n{header_context}\n"
                    chunks.append(
                        CodeChunk(
                            file_path=file_path,
                            chunk_type=node.type,
                            name=name,
                            start_line=node.start_point[0] + 1,
                            end_line=node.end_point[0] + 1,
                            content=node_text,
                            context=context,
                        )
                    )
            else:
                for child in node.children:
                    traverse(child, scope_prefix)

        traverse(root_node)

        # 兜底：若文件没有提取出 AST 结构（如纯配置或脚本），回退到滑动窗口分块
        if not chunks:
            return self._fallback_line_split(file_path, code_content, header_context)

        return chunks

    def _get_node_name(self, node, code: str) -> str:
        for child in node.children:
            if child.type in {"identifier", "name"}:
                return code[child.start_byte : child.end_byte]
        return "anonymous"

    def _extract_header_imports(self, root_node, code: str, max_lines: int = 15) -> str:
        """提取头部 import / using 语句（截取前 N 行）"""
        imports = []
        for child in root_node.children:
            if "import" in child.type or "use" in child.type or "package" in child.type:
                imports.append(code[child.start_byte : child.end_byte])
            if len(imports) >= max_lines:
                break
        return "\n".join(imports)

    def _fallback_line_split(
        self, file_path: str, code: str, header: str
    ) -> List[CodeChunk]:
        """对于平铺脚本或配置文件的简单行切分"""
        lines = code.splitlines()
        chunks = []
        step = 50
        for i in range(0, len(lines), step):
            chunk_lines = lines[i : i + step + 10]
            chunks.append(
                CodeChunk(
                    file_path=file_path,
                    chunk_type="block",
                    name=f"lines_{i+1}_{i+len(chunk_lines)}",
                    start_line=i + 1,
                    end_line=i + len(chunk_lines),
                    content="\n".join(chunk_lines),
                    context=f"// File: {file_path}\n{header}\n",
                )
            )
        return chunks

## 封装自定义 Embedding 客户端

In [ ]:
import os
from openai import OpenAI
from typing import List

class CustomOpenAIEmbedding:
    def __init__(
        self,
        base_url: str = os.environ['EMBEDDING_BASE_URL'], 
        api_key: str = os.environ['EMBEDDING_API_KEY'],
        model: str = os.environ['EMBEDDING_MODEL']
    ):
        self.client = OpenAI(base_url=base_url, api_key=api_key, timeout=10 * 60)
        self.model = model
        self._dim = None

    def embed_documents(self, texts: List[str], batch_size: int = 64) -> List[List[float]]:
        """批量计算文档嵌入向量"""
        all_embeddings = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            response = self.client.embeddings.create(
                input=batch,
                model=self.model
            )
            # 保证返回顺序与输入一致
            sorted_data = sorted(response.data, key=lambda x: x.index)
            all_embeddings.extend([item.embedding for item in sorted_data])
            
        if not self._dim and all_embeddings:
            self._dim = len(all_embeddings[0])
        return all_embeddings

    def embed_query(self, text: str) -> List[float]:
        """计算单条查询语句的嵌入向量"""
        response = self.client.embeddings.create(
            input=[text],
            model=self.model
        )
        return response.data[0].embedding

    @property
    def dimension(self) -> int:
        """获取向量维度（动态探测或手动指定）"""
        if self._dim is None:
            self._dim = len(self.embed_query("probe"))
        return self._dim

## 混合索引实现 (Vector + BM25/Symbol)

In [4]:
import uuid
from typing import List, Dict, Any
from qdrant_client import QdrantClient, models
from fastembed import SparseTextEmbedding

class QdrantCodeHybridRAG:
    def __init__(
        self,
        collection_name: str = "code_hybrid_kb",
        qdrant_path: str = "./qdrant_data"  # 本地持久化目录，或传 url="http://localhost:6333"
    ):
        self.collection_name = collection_name
        
        # 1. 初始化客户端与模型
        self.client = QdrantClient(path=qdrant_path)
        self.dense_model = CustomOpenAIEmbedding()
        # BM25 稀疏模型 (生成 词索引 -> 权重 的 Sparse Vector)
        self.sparse_model = SparseTextEmbedding(model_name="Qdrant/bm25")

        # 2. 初始化双向量集合
        self._setup_collection()

    def _setup_collection(self):
        """定义包含 Dense 和 Sparse 两种向量命名的 Schema"""
        if not self.client.collection_exists(self.collection_name):
            self.client.create_collection(
                collection_name=self.collection_name,
                vectors_config={
                    "dense_vector": models.VectorParams(
                        size=self.dense_model.dimension,
                        distance=models.Distance.COSINE
                    )
                },
                sparse_vectors_config={
                    "sparse_vector": models.SparseVectorParams(
                        index=models.SparseIndexParams(on_disk=False)
                    )
                }
            )
            print(f"已成功创建集合: {self.collection_name}")
            
            # 【关键步骤】为 file_path 创建 Keyword 索引，加速按文件路径精确删除/过滤
            self.client.create_payload_index(
                collection_name=self.collection_name,
                field_name="file_path",
                field_schema=models.PayloadSchemaType.KEYWORD
            )

    def delete_by_file_path(self, file_path: str):
        """1. 物理删除指定文件路径下的所有旧 Chunks"""
        self.client.delete(
            collection_name=self.collection_name,
            points_selector=models.FilterSelector(
                filter=models.Filter(
                    must=[
                        models.FieldCondition(
                            key="file_path",
                            match=models.MatchValue(value=file_path)
                        )
                    ]
                )
            )
        )
        print(f"[清理完成] 已移除 {file_path} 的所有历史数据")

    def index_code_chunks(self, code_chunks: List[CodeChunk]):
        """
        批量计算 Dense 与 Sparse 向量并写入 Qdrant
        """
        texts = [f"{chunk['context']}\n{chunk['content']}" for chunk in code_chunks]

        # 1. 批量计算 Dense 向量
        dense_embeddings = self.dense_model.embed_documents(texts)

        # 2. 批量计算 Sparse (BM25) 向量
        sparse_embeddings = list(self.sparse_model.embed(texts))

        # 3. 构造 PointStruct 并上传
        points = []
        for i, chunk in enumerate(code_chunks):
            # 将 FastEmbed 输出转换为 Qdrant 兼容的 SparseVector 格式
            sparse_val = sparse_embeddings[i]
            sparse_vector = models.SparseVector(
                indices=sparse_val.indices.tolist(),
                values=sparse_val.values.tolist()
            )

            point = models.PointStruct(
                id=str(uuid.uuid4()),
                vector={
                    "dense_vector": dense_embeddings[i],
                    "sparse_vector": sparse_vector
                },
                payload=chunk
            )
            points.append(point)

        self.client.upsert(collection_name=self.collection_name, points=points)
        print(f"成功索引 {len(points)} 个代码片段")
        
    def index_directory(self, repo_dir: str, clean_existing: bool = True):
        """
        一键遍历本地代码库并构建/更新混合索引
        """
        loader = CodebaseLoader(base_dir=repo_dir)
        print(f"正在扫描并切分目录: {repo_dir} ...")
        chunks = loader.load_and_split_all()
        
        if not chunks:
            print("未找到需要索引的代码文件。")
            return

        # 获取所有涉及到的文件路径
        file_paths = {chunk["file_path"] for chunk in chunks}
        print(f"扫描到 {len(file_paths)} 个文件，共切分为 {len(chunks)} 个代码块")

        # 如果需要先清理已有数据（避免同一文件修改后产生重复历史分块）
        if clean_existing:
            print("正在清理旧文件索引...")
            for fp in file_paths:
                self.delete_by_file_path(fp)

        # 批量构建索引
        self.index_code_chunks(chunks)
        print(f"代码库 {repo_dir} 索引构建完成！")


    def hybrid_search(self, query: str, top_k: int = 3) -> List[CodeChunk]:
        """
        执行双路召回并通过 RRF (Reciprocal Rank Fusion) 自动融合打分
        """
        # 1. 计算 Query 的 Dense 向量
        dense_query = self.dense_model.embed_query(query)

        # 2. 计算 Query 的 Sparse BM25 向量
        sparse_query_obj = list(self.sparse_model.embed([query]))[0]
        sparse_query = models.SparseVector(
            indices=sparse_query_obj.indices.tolist(),
            values=sparse_query_obj.values.tolist()
        )

        # 3. 使用 query_points 进行 Prefetch + RRF 融合
        search_result = self.client.query_points(
            collection_name=self.collection_name,
            prefetch=[
                # Dense 路：语义召回
                models.Prefetch(
                    query=dense_query,
                    using="dense_vector",
                    limit=top_k * 3
                ),
                # Sparse 路：精确关键词/符号 BM25 召回
                models.Prefetch(
                    query=sparse_query,
                    using="sparse_vector",
                    limit=top_k * 3
                ),
            ],
            # RRF (倒数排名融合)：根据两路召回的排名综合打分
            query=models.FusionQuery(fusion=models.Fusion.RRF),
            limit=top_k
        )

        # 4. 解析结果
        results = []
        for point in search_result.points:
            results.append({
                "score": point.score,
                **(point.payload or {})
            })
        return results

c:\Users\zengd\Desktop\lc_agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import os, traceback
from pathlib import Path
from typing import List, Dict, Set, Generator

class CodebaseLoader:
    # 默认支持的代码后缀与对应 tree-sitter 解析语言
    DEFAULT_EXT_TO_LANG = {
        ".py": "python",
        ".ts": "typescript",
        ".tsx": "typescript",
        ".js": "typescript",
        ".jsx": "typescript",
        ".java": "java",
        # 兜底纯文本/脚本文件（将回退到行切分）
        ".md": "text",
        ".json": "text",
        ".yaml": "text",
        ".yml": "text",
        ".sql": "text",
    }

    # 默认忽略的目录名
    DEFAULT_IGNORE_DIRS: Set[str] = {
        ".git", ".svn", ".hg",
        ".venv", "venv", "env", ".env",
        "node_modules", "bower_components",
        "__pycache__", ".pytest_cache", ".mypy_cache", ".ruff_cache",
        ".idea", ".vscode",
        "dist", "build", "out", "target", "bin", "obj",
        "qdrant_data", "qdrant_hybrid_storage"
    }

    # 默认忽略的文件名或后缀
    DEFAULT_IGNORE_EXTS: Set[str] = {
        ".pyc", ".pyo", ".pyd", ".so", ".dll", ".dylib", ".exe",
        ".png", ".jpg", ".jpeg", ".gif", ".ico", ".svg", ".webp",
        ".zip", ".tar", ".gz", ".7z", ".rar",
        ".db", ".sqlite", ".sqlite3",
        ".min.js", ".min.css", ".map"
    }

    def __init__(
        self,
        base_dir: str,
        ext_to_lang: Dict[str, str] = None,
        ignore_dirs: Set[str] = None,
        ignore_exts: Set[str] = None,
        max_file_size_kb: int = 500  # 超过 500KB 的文件通常是打包产物/大数据文件，直接跳过
    ):
        self.base_dir = Path(base_dir).resolve()
        self.ext_to_lang = ext_to_lang or self.DEFAULT_EXT_TO_LANG
        self.ignore_dirs = ignore_dirs or self.DEFAULT_IGNORE_DIRS
        self.ignore_exts = ignore_exts or self.DEFAULT_IGNORE_EXTS
        self.max_file_size_bytes = max_file_size_kb * 1024

    def is_ignored(self, path: Path) -> bool:
        """检查文件或目录是否应该被忽略"""
        # 1. 检查各级目录是否在忽略名单中
        for part in path.parts:
            if part in self.ignore_dirs or (part.startswith(".") and part not in {".", ".."} and part in self.ignore_dirs):
                return True

        # 2. 检查文件后缀
        if path.suffix.lower() in self.ignore_exts:
            return True

        # 3. 如果设置了白名单后缀且当前后缀不在其中
        if self.ext_to_lang and path.suffix.lower() not in self.ext_to_lang:
            return True

        # 4. 检查文件大小
        if path.is_file() and path.stat().st_size > self.max_file_size_bytes:
            return True

        return False

    def walk_files(self) -> Generator[Path, None, None]:
        """递归遍历目录，产出有效代码文件路径"""
        for root, dirs, files in os.walk(self.base_dir, topdown=True):
            root_path = Path(root)

            # 原地修改 dirs 以跳过忽略的目录，避免无谓的递归下钻
            dirs[:] = [d for d in dirs if d not in self.ignore_dirs and not d.startswith(".git")]

            for file_name in files:
                file_path = root_path / file_name
                if not self.is_ignored(file_path):
                    yield file_path

    def load_and_split_all(self, max_chunk_size: int = 1500) -> List[Dict]:
        """遍历整个目录并使用 AST 切分所有文件"""
        all_chunks = []
        # 缓存不同语言的 AST 切分器实例
        splitters = {}

        for file_path in self.walk_files():
            # 获取相对于 base_dir 的相对路径，方便作为唯一标识和存储展示
            rel_path = file_path.relative_to(self.base_dir).as_posix()
            lang = self.ext_to_lang.get(file_path.suffix.lower())

            try:
                content = file_path.read_text(encoding="utf-8", errors="ignore")
                if not content.strip():
                    continue

                # 获取或创建该语言的 Splitter
                if lang not in splitters:
                    splitters[lang] = CodeASTSplitter(language=lang, max_chunk_size=max_chunk_size)
                    
                chunks = splitters[lang].split_file(file_path=rel_path, code_content=content)
                # 转换 dataclass 为 dict 格式
                for c in chunks:
                    all_chunks.append(c.__dict__ if hasattr(c, "__dict__") else c)

            except Exception as e:
                print(f"[跳过] 解析文件失败 {rel_path}")
                traceback.print_exc()

        return all_chunks


## 测试

In [ ]:
from dotenv import load_dotenv

load_dotenv()

# 初始化本地 Hybrid RAG 实例
rag = QdrantCodeHybridRAG(
    collection_name="demo_repo_hybrid", qdrant_path="../qdrant_hybrid_storage"
)

# 准备测试代码数据
sample_code_chunks = [
    {
        "file_path": "src/auth/jwt_handler.py",
        "name": "verify_session_token",
        "content": "def verify_session_token(token: str) -> bool:\n    # 校验用户 JWT 是否过期\n    return decode_jwt(token).is_valid()",
        "context": "# File: src/auth/jwt_handler.py\n# Symbol: verify_session_token\n",
    },
    {
        "file_path": "src/database/connection.py",
        "name": "get_db_pool",
        "content": "def get_db_pool():\n    # 创建数据库连接池\n    return create_engine('postgresql://localhost:5432/db')",
        "context": "# File: src/database/connection.py\n# Symbol: get_db_pool\n",
    },
    {
        "file_path": "src/utils/errors.py",
        "name": "InvalidSignatureError",
        "content": 'class InvalidSignatureError(Exception):\n    """签名校验失败异常"""\n    pass',
        "context": "# File: src/utils/errors.py\n# Symbol: InvalidSignatureError\n",
    },
]

file_paths = {chunk["file_path"] for chunk in sample_code_chunks}
for file_path in file_paths:
    rag.delete_by_file_path(file_path)

# 1. 写入混合索引
rag.index_code_chunks(sample_code_chunks)

# 2. 混合测试 A：语义模糊查询 (Dense 占主导)
print("\n--- 语义意图测试: '检查登录凭证是否有效' ---")
results_a = rag.hybrid_search("检查登录凭证是否有效", top_k=1)
for res in results_a:
    print(
        f"Score: {res['score']:.4f} | File: {res['file_path']} | Symbol: {res['name']}"
    )

# 3. 混合测试 B：精确符号查询 (Sparse BM25 占主导)
print("\n--- 关键词符号测试: 'InvalidSignatureError' ---")
results_b = rag.hybrid_search("InvalidSignatureError", top_k=1)
for res in results_b:
    print(
        f"Score: {res['score']:.4f} | File: {res['file_path']} | Symbol: {res['name']}"
    )

已成功创建集合: demo_repo_hybrid
[清理完成] 已移除 src/database/connection.py 的所有历史数据
[清理完成] 已移除 src/utils/errors.py 的所有历史数据
[清理完成] 已移除 src/auth/jwt_handler.py 的所有历史数据


C:\Users\zengd\AppData\Local\Temp\ipykernel_2300\761509657.py:43: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  self.client.create_payload_index(


成功索引 3 个代码片段

--- 语义意图测试: '检查登录凭证是否有效' ---
Score: 0.5000 | File: src/auth/jwt_handler.py | Symbol: verify_session_token

--- 关键词符号测试: 'InvalidSignatureError' ---
Score: 1.0000 | File: src/utils/errors.py | Symbol: InvalidSignatureError


In [ ]:
# 初始化 RAG
rag = QdrantCodeHybridRAG(
    collection_name="my_project_kb",
    qdrant_path="../qdrant_hybrid_storage"
)

# 索引当前工作区下的某个项目或文件夹
# （例如遍历当前项目的根目录 "."）
loader = CodebaseLoader(base_dir="C:\\Users\\zengd\\Desktop\\lc_agent\\src")
chunks = loader.load_and_split_all()

print(f"共提取到 {len(chunks)} 个代码分块")
for chunk in chunks:
    print(f"-> 文件: {chunk['file_path']}, 符号: {chunk['name']}, 行号: {chunk['start_line']}-{chunk['end_line']}")

# 写入 Qdrant 索引
rag.index_code_chunks(chunks)


共提取到 12 个代码分块
-> 文件: app.py, 符号: gent() -> Compil, 行号: 18-22
-> 文件: agent/index.py, 符号: get_agent, 行号: 14-27
-> 文件: agent/index.py, 符号: main, 行号: 30-48
-> 文件: mcp_tools/index.py, 符号: get_mcp_tools, 行号: 5-32
-> 文件: prompt/index.py, 符号: lines_1_60, 行号: 1-60
-> 文件: prompt/index.py, 符号: lines_51_90, 行号: 51-90
-> 文件: skills/index.py, 符号: parse_skill_meta, 行号: 9-23
-> 文件: skills/index.py, 符号: 所有已安装技能的索引目录（仅包含技能名称和, 行号: 30-46
-> 文件: skills/index.py, 符号: (target_pa, 行号: 50-60
-> 文件: skills/index.py, 符号: , 行号: 64-71
-> 文件: tools/index.py, 符号: run_shell_command, 行号: 9-41
-> 文件: tools/index.py, 符号: , 行号: 45-47


InternalServerError: <!DOCTYPE HTML PUBLIC "-//IETF//DTD HTML 2.0//EN">
<html>
<head><title>504 Gateway Time-out</title></head>
<body>
<center><h1>504 Gateway Time-out</h1></center>
<hr/>Powered by Tengine<hr><center>nginx</center>
</body>
</html>

: 